[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/history/h06_ramanujan_pi.ipynb)

# Ramanujan's formula for π: eight digits a step

In 1914 Srinivasa Ramanujan published a formula for π with the number 9801 in it, and no proof. Nobody proved it for
73 years. Its secret is speed: every term of its sum adds about eight more correct digits of π. Here you can race it
against Madhava's slow sum, watch the digits lock in, find the 99 hidden in it, meet its faster cousin (the formula
behind today's records), and try the challenge from the video.

**How to use this notebook.** The grey boxes are cells. Click the first one, then press **Shift + Enter** (or the
▶ Run button) to run it. Run each cell in turn, from the top to the bottom. When a slider appears, drag it and watch.
Change any number you like and run the cell again. Nothing can break: if something looks odd, run everything again
from the top. It works in your browser (no install) and in Google Colab.

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look, and a few helpers. Run this cell first.
import math
from decimal import Decimal, getcontext
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, ToggleButton

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM = "#ece6d6", "#93a097"           # chalk, and chalk smudge
CORAL = "#e3896b"                           # the one key thing in each figure
TEAL, SAGE, LILAC = "#6fb6bb", "#a0c27c", "#c9a0d3"

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#3b4c47", "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "figure.dpi": 100,
})

getcontext().prec = 120                     # we work with 120 significant digits
PI = Decimal("3.14159265358979323846264338327950288419716939937510582097494459230781640628620899862803482534211706798214808651")

def matching(x):
    """How many decimals of x agree with π, digit by digit."""
    a, b = f"{x:.100f}"[2:], f"{PI:.100f}"[2:]
    return next((i for i, (p, q) in enumerate(zip(a, b)) if p != q), 100)

print("Ready. Run the next cell.")

## 1 · Madhava's sum: beautiful, and slow

π = 4 × (1 − 1/3 + 1/5 − 1/7 + …). Pick how many terms to add and see how close you get. Try 10, 100, 1,000, then
100,000.

In [ ]:
def madhava(n):
    return 4 * sum((-1) ** k / (2 * k + 1) for k in range(n))

def slow(n=10):
    v = madhava(n)
    print(f"{n:,} terms: {v:.10f}")
    print(f"π       : {math.pi:.10f}")
    print(f"correct decimals: {matching(Decimal(repr(v)))}")
    ks = list(range(1, min(n, 60) + 1))
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.plot(ks, [madhava(k) for k in ks], color=TEAL, marker="o", ms=3, lw=1.2, label="running total")
    ax.axhline(math.pi, color=SAGE, lw=1.5, label="π")
    ax.set_xlabel("terms"); ax.set_ylim(2.6, 4.1); ax.legend(); plt.show()

interact(slow, n=IntSlider(value=10, min=1, max=100_000, step=1, description="terms"));

## 2 · Ramanujan's sum, term by term

1/π = (2√2 / 9801) × Σ (4k)! (1103 + 26390k) / ((k!)⁴ × 396⁴ᵏ), adding up k = 0, 1, 2, …

Add one term at a time and count the correct decimals. The coral digit is the first wrong one.

In [ ]:
def ramanujan(n):
    s = sum(Decimal(math.factorial(4 * k) * (1103 + 26390 * k)) / Decimal(math.factorial(k) ** 4 * 396 ** (4 * k))
            for k in range(n))
    return 1 / (2 * Decimal(2).sqrt() / 9801 * s)

def fast(n=1):
    for k in range(1, n + 1):
        x = ramanujan(k); m = matching(x)
        print(f"{k} term{'s' if k > 1 else ' '}: {str(x)[:2 + m]}[{str(x)[2 + m]}]…   {m} correct decimals")
    print(f"π      : {str(PI)[:60]}…")

interact(fast, n=IntSlider(value=1, min=1, max=10, description="terms"));

## 3 · Why eight digits a term? The 99 hidden inside

Each term is the last one divided by a big number. That number settles down towards 99⁴ = 96,059,601, about 10⁸,
and 10⁸ is a one followed by eight zeros. And look where 99 hides: 9801 = 99 × 99 and 396 = 4 × 99.

In [ ]:
terms = [Decimal(math.factorial(4 * k) * (1103 + 26390 * k)) / Decimal(math.factorial(k) ** 4 * 396 ** (4 * k))
         for k in range(12)]
for k in range(11):
    r = terms[k] / terms[k + 1]
    print(f"term {k + 1} ÷ term {k + 2} = {float(r):>14,.0f}   ≈ 10^{math.log10(r):.2f}")
print(f"\n99⁴ = {99 ** 4:,}    9801 = 99 × 99 = {99 * 99}    396 = 4 × 99 = {4 * 99}")

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(range(1, 13), [math.log10(t) for t in terms], color=LILAC, marker="o", lw=2)
ax.set_xlabel("term"); ax.set_ylabel("size (power of ten)"); ax.set_title("each step: about 8 powers of ten smaller")
plt.show()

## 4 · The cousin behind the records

In 1988 David and Gregory Chudnovsky built a formula the same way:
1/π = 12 Σ (−1)ᵏ (6k)! (13591409 + 545140134k) / ((3k)! (k!)³ 640320^(3k + 3/2)).
Each term adds about 14 digits. The 314-trillion-digit record of 2025 ran on it.

In [ ]:
def chudnovsky(n):
    c = Decimal(640320) ** 3
    s = sum(Decimal((-1) ** k * math.factorial(6 * k) * (13591409 + 545140134 * k)) /
            Decimal(math.factorial(3 * k) * math.factorial(k) ** 3) / c ** k for k in range(n))
    return Decimal(640320) ** Decimal("1.5") / (12 * s)

for n in range(1, 7):
    print(f"{n} term{'s' if n > 1 else ' '}: {matching(chudnovsky(n))} correct decimals")
print(f"\ndigits a term: log10(640320³ / 1728) = {math.log10(640320 ** 3 / 1728):.2f}")
print(f"314 trillion digits ÷ 14.18 ≈ {314e12 / math.log10(640320 ** 3 / 1728) / 1e12:.1f} trillion terms")

## 5 · A number that is almost whole

Raise e to the power π√58. Ramanujan's numbers come from here: the answer is within a millionth of 396⁴ − 104.

In [ ]:
x = (PI * Decimal(58).sqrt()).exp()
print(f"e^(π√58)   = {x:.15f}")
print(f"396⁴ − 104 = {396 ** 4 - 104:,}")
print(f"difference = {396 ** 4 - 104 - x:.3e}")

## 6 · The challenge from the video

In the same 1914 paper Ramanujan noticed that π⁴ is very close to 2143 / 22. Take the fourth root: how many decimals
of π do you get? Think first (or try it on a calculator), then press the button.

In [ ]:
def challenge(show_answer=False):
    if not show_answer:
        print("(2143 / 22) to the power 1/4: how many decimals of π?   (press the button when you're ready)"); return
    r = (Decimal(2143) / 22) ** (Decimal(1) / 4)
    print(f"(2143/22)^(1/4) = {str(r)[:16]}…")
    print(f"π               = {str(PI)[:16]}…")
    print(f"they agree to {matching(r)} decimals")

interact(challenge, show_answer=ToggleButton(value=False, description="show the answer"));

## Sources
S. Ramanujan, "Modular equations and approximations to π", *Quarterly Journal of Mathematics* 45 (1914) 350–372.
J. M. Borwein and P. B. Borwein, *Pi and the AGM* (1987). D. V. and G. V. Chudnovsky (1988). Wikipedia: Chronology of
computation of π; Chudnovsky algorithm; Madhava series. y-cruncher records (314 trillion digits, StorageReview,
November 2025). All retrieved 2026-10-30. Every number above is computed in the cells.